In [1]:
import pandas as pd
import numpy as np

def calculate_kpi_and_label(df_input):
    """
    Calculates the Performance Premium KPI (Price / Engine Size), 
    dynamically applies labels based on quartiles, groups by Model, and finds the top 10 for the report.
    """
    
    TARGET_REGION = 'Europe'
    TARGET_YEAR = 2024
    
    df_filtered = df_input[
        (df_input['Region'] == TARGET_REGION) & 
        (df_input['Year'] == TARGET_YEAR) # CORRECTED: Changed df_filtered to df_input here
    ].copy()
    
    print(f"Data filtered for {TARGET_REGION} in {TARGET_YEAR}. Rows: {len(df_filtered)}")
    
    df_filtered['Engine_Size_L_Safe'] = df_filtered['Engine_Size_L'].replace(0, 0.1)
    df_filtered['Performance_Premium'] = df_filtered['Price_USD'] / df_filtered['Engine_Size_L_Safe']
    
    df_grouped_model = df_filtered.groupby('Model').agg(
        Price_USD=('Price_USD', 'mean'),
        Engine_Size_L=('Engine_Size_L', 'mean'),
        Performance_Premium=('Performance_Premium', 'mean')
    ).reset_index()

    kpi_quartiles = df_grouped_model['Performance_Premium'].quantile([0.25, 0.50, 0.75])
    Q1 = kpi_quartiles.loc[0.25]
    Q3 = kpi_quartiles.loc[0.75]
    
    print(f"Calculated Quartiles (Model Averages): Q1={Q1:.0f}, Median/Q2={kpi_quartiles.loc[0.50]:.0f}, Q3={Q3:.0f}")
    
    def label_kpi_dynamic(kpi_value, q1, q3):
        """Labels the KPI based on dynamic quartile boundaries."""
        if kpi_value > q3:
            return 'High (Top 25%)'
        elif kpi_value > q1:
            return 'Medium (Mid 50%)'
        else:
            return 'Low (Bottom 25%)'
    df_grouped_model['KPI_Label'] = df_grouped_model['Performance_Premium'].apply(
        lambda x: label_kpi_dynamic(x, Q1, Q3)
    )
    df_sorted = df_grouped_model.sort_values(
        by='Performance_Premium', 
        ascending=False
    )
    
    top_10 = df_sorted.head(10).copy()

    final_table = top_10[[
        'Model',
        'Price_USD',
        'Engine_Size_L',
        'Performance_Premium',
        'KPI_Label'
    ]].copy()
    final_table['Price_USD'] = final_table['Price_USD'].round(0).astype(int)
    final_table['Engine_Size_L'] = final_table['Engine_Size_L'].round(2)
    final_table['Performance_Premium'] = final_table['Performance_Premium'].round(0).astype(int)
    final_table.columns = [
        'Model (ID/Name)', 'Price_USD (Input1 - Avg)', 'Engine_Size_L (Input2 - Avg)', 'Performance_Premium (KPI)','Label']
    
    print("\nTop 10 Performance Premium Models (Europe, 2024)\n")
    
    # Return statement is simplified
    return final_table

# --- Execution Block ---
try:
    df_consolidated = pd.read_csv('bmw_consolidated_data.csv')
except FileNotFoundError:
    try:
        # Fallback to original file
        df_consolidated = pd.read_csv('BMW sales data (2010-2024) (1).csv')
        print("Using original data as 'bmw_consolidated_data.csv' was not found.")
    except FileNotFoundError:
        print("Fatal Error: Original data file not found either. Cannot proceed.")
        exit()

# The function call handles the new return structure
calculate_kpi_and_label(df_consolidated)

Data filtered for Europe in 2024. Rows: 592
Calculated Quartiles (Model Averages): Q1=24359, Median/Q2=25408, Q3=27468

Top 10 Performance Premium Models (Europe, 2024)



,Model (ID/Name),Price_USD (Input1 - Avg),Engine_Size_L (Input2 - Avg),Performance_Premium (KPI),Label
6,X3,74936,3.10,27797,High (Top 25%)
1,5 Series,77825,3.13,27713,High (Top 25%)
0,3 Series,77384,3.10,27666,High (Top 25%)
10,i8,75216,3.11,27270,Medium (Mid 50%)
4,M5,73727,3.22,26009,Medium (Mid 50%)
5,X1,69829,3.19,25408,Medium (Mid 50%)
7,X5,68769,3.07,25216,Medium (Mid 50%)
2,7 Series,67037,3.19,24623,Medium (Mid 50%)
3,M3,71025,3.26,24095,Low (Bottom 25%)
8,X6,69839,3.24,23288,Low (Bottom 25%)


In [26]:
import pandas as pd

def make_cohorts(file_name):
    df = pd.read_csv(file_name)
    df['clean_model'] = df['Model'].str.strip().str.lower()
    cohorts = {
        'X_models': df['clean_model'].str.startswith('x'),
        'M_models': df['clean_model'].str.startswith('m'),
        'Series_models': df['clean_model'].str.contains('series')
    }
    summary_list = []
    
    for name, filter_mask in cohorts.items():
        cohort_df = df[filter_mask]
        count = len(cohort_df)
        
        examples = cohort_df['Model'].head(5).tolist()
        
        summary_list.append({
            'Cohort Name': name,
            'Count': count,
            'First 5 Examples': ", ".join(examples)
        })

    final_table = pd.DataFrame(summary_list)
    print("Cohort Analysis Summary (Model Cohorts)")
    print("\n")
    print(final_table.to_string(index=False))
    print("\n")
    

file_to_use = 'BMW sales data (2010-2024) (1).csv'

make_cohorts(file_to_use)

Cohort Analysis Summary (Model Cohorts)


  Cohort Name  Count                                 First 5 Examples
     X_models  18032                               X3, X3, X1, X5, X5
     M_models   8891                               M5, M3, M5, M5, M3
Series_models  13853 5 Series, 5 Series, 7 Series, 5 Series, 3 Series




In [25]:
import pandas as pd
import numpy as np

def create_decision_matrix(input_filename, price_budget_usd):
    """
    Analyzes Asia 2024 models based on availability, price, and KPI performance 
    and generates the requested 2D decision matrix table.
    """
    
    TARGET_REGION = 'Asia'
    TARGET_YEAR = 2024
    MAX_PRICE_USD = price_budget_usd
    
    df = pd.read_csv(input_filename)
    df_filtered = df[(df['Region'] == TARGET_REGION) & (df['Year'] == TARGET_YEAR)].copy()
    
    if df_filtered.empty:
        print(f"No data found for {TARGET_REGION} in {TARGET_YEAR}.")
        return

    df_filtered['Engine_Size_L_Safe'] = df_filtered['Engine_Size_L'].replace(0, 0.1)
    df_filtered['Performance_Premium'] = df_filtered['Price_USD'] / df_filtered['Engine_Size_L_Safe']

    df_grouped = df_filtered.groupby('Model').agg(
        Avg_Price_USD=('Price_USD', 'mean'),
        Avg_KPI=('Performance_Premium', 'mean'),
        Has_Black=('Color', lambda x: 'Black' in x.values),
    ).reset_index()

    Q1_kpi_threshold = df_grouped['Avg_KPI'].quantile(0.25)

    print(f"Analysis for: {TARGET_REGION}, {TARGET_YEAR}")
    print(f"Dynamic KPI Threshold (Q1/Low boundary): {Q1_kpi_threshold:.0f} USD/L")
    print(f"Dynamic Price Budget (Max): ${MAX_PRICE_USD:,.0f} USD")
    
    df_grouped['Black Available'] = df_grouped['Has_Black'].apply(lambda x: 'Yes' if x else 'No')
    
    df_grouped[f'Price Below ${MAX_PRICE_USD:,.0f}'] = np.where(
        df_grouped['Avg_Price_USD'] <= MAX_PRICE_USD, 'Yes', 'No'
    )

    df_grouped[f'KPI Medium or High (> Q1)'] = np.where(
        df_grouped['Avg_KPI'] >= Q1_kpi_threshold, 'Yes', 'No'
    )
    
    decision_matrix = df_grouped[[
        'Model',
        'Black Available',
        f'Price Below ${MAX_PRICE_USD:,.0f}',
        f'KPI Medium or High (> Q1)'
    ]].copy()

    decision_matrix.columns.values[0] = 'Model'

    print("\n----------------------------------------------------")
    print("Decision Matrix: Model Suitability (Asia, 2024)")
    print("----------------------------------------------------")
    print(decision_matrix.to_string(index=False))
    print("\n")
    
    return df_filtered, decision_matrix

# --- Execution Block ---
# Setting the dynamic price budget here
PRICE_BUDGET = 75000 
input_file = 'BMW sales data (2010-2024) (1).csv'

# We call the function to create and print the 2D matrix
df_filtered_rows, decision_matrix = create_decision_matrix(input_file, PRICE_BUDGET)

Analysis for: Asia, 2024
Dynamic KPI Threshold (Q1/Low boundary): 25399 USD/L
Dynamic Price Budget (Max): $75,000 USD

----------------------------------------------------
Decision Matrix: Model Suitability (Asia, 2024)
----------------------------------------------------
   Model Black Available Price Below $75,000 KPI Medium or High (> Q1)
3 Series             Yes                  No                       Yes
5 Series             Yes                 Yes                        No
7 Series             Yes                  No                       Yes
      M3             Yes                  No                       Yes
      M5             Yes                 Yes                       Yes
      X1             Yes                  No                       Yes
      X3             Yes                 Yes                        No
      X5             Yes                  No                       Yes
      X6             Yes                  No                       Yes
      i3         